In [4]:
import pandas as pd
from langdetect import detect
import re

In [ ]:
import pandas as pd
from langdetect import detect
from typing import List

# -------- Language Utilities -------- #

def is_not_target_language(text: str, lang_code: str) -> bool:
    """Return True if detected language is not the target language."""
    try:
        return detect(text) != lang_code
    except:
        return True

def looks_like_mistranslation(src: str, tgt: str, length_factor: float = 2.0) -> bool:
    """Flag if translation is suspiciously longer than source."""
    return len(src.split()) <= 10 and len(tgt.split()) > length_factor * len(src.split())

# -------- Dataset Cleanup -------- #

def flag_mt_issues(df: pd.DataFrame, target_lang_code: str) -> pd.DataFrame:
    """Add columns to flag non-target-language and suspicious translations."""
    df = df.copy()
    df["flag_non_target_lang"] = df["targets"].apply(lambda text: is_not_target_language(text, target_lang_code))
    df["flag_mistranslation"] = df.apply(lambda row: looks_like_mistranslation(row["inputs"], row["targets"]), axis=1)
    df["needs_review"] = df["flag_non_target_lang"] | df["flag_mistranslation"]
    return df

# -------- Bootstrap Examples -------- #

def create_simple_mt_examples(language: str) -> pd.DataFrame:
    """Return a few high-quality examples for English → target language MT."""
    examples = {
        "hausa": {
            "inputs": [
                "Good morning.",
                "Thank you very much.",
                "I am learning Hausa.",
                "What is your name?",
                "Where are you going?"
            ],
            "targets": [
                "Ina kwana.",
                "Na gode sosai.",
                "Ina koyon Hausa.",
                "Menene sunanka?",
                "Ina za ka je?"
            ]
        },
        "swahili": {
            "inputs": [
                "Good morning.",
                "Thank you very much.",
                "I am learning Swahili.",
                "What is your name?",
                "Where are you going?"
            ],
            "targets": [
                "Habari za asubuhi.",
                "Asante sana.",
                "Ninajifunza Kiswahili.",
                "Jina lako nani?",
                "Unaenda wapi?"
            ]
        }
    }
    data = examples[language.lower()]
    return pd.DataFrame({
        "ID": [f"bootstrap_{language}_{i}" for i in range(len(data["inputs"]))],
        "task": ["mmt"] * len(data["inputs"]),
        "langs": ["eng-" + language[:3]] * len(data["inputs"]),
        "data_source": ["synthetic"] * len(data["inputs"]),
        "instruction": ["Translate the following from English into {}.".format(language.capitalize())] * len(data["inputs"]),
        "inputs": data["inputs"],
        "targets": data["targets"]
    })

# -------- Standardization + Save -------- #

def clean_mt_dataset(
    df: pd.DataFrame,
    target_lang: str,
    lang_code: str,
    standard_instruction: str,
    output_file: str,
    include_flagged_csv: bool = True
) -> pd.DataFrame:
    """Clean MT dataset and save improved version."""
    
    # Flag issues
    df = flag_mt_issues(df, target_lang_code=lang_code)

    # Standardize instruction
    df["instruction"] = standard_instruction

    # Filter clean rows
    clean_df = df[~df["needs_review"]].copy()

    # Add simple high-quality examples
    boot_df = create_simple_mt_examples(target_lang)
    final_df = pd.concat([clean_df, boot_df], ignore_index=True)

    # Save final dataset
    final_df.to_csv(output_file, index=False)
    print(f"✅ Cleaned dataset saved to '{output_file}' with {len(final_df)} rows.")

    if include_flagged_csv:
        flagged_path = output_file.replace(".csv", "_flagged.csv")
        df[df["needs_review"]].to_csv(flagged_path, index=False)
        print(f"⚠️ Flagged rows saved to '{flagged_path}' for manual review.")

    return final_df
